# Tiền xử lý & chuẩn hoá bộ dữ liệu giá BĐS TP.HCM

**Nguồn:** `df_raw` — 11888 tin rao bán tại TP.HCM (batdongsan + cafeland)

**Tổng quan dự án tiền xử lý**
Mục tiêu: Chuẩn hóa dữ liệu thô, loại bỏ nhiễu và khai phá các đặc trưng ẩn từ văn bản (tiêu đề và mô tả) để phục vụ cho các bước phân tích hoặc xây dựng mô hình tiếp theo.

Đầu ra: Bảng dữ liệu hoàn thiện df_clean với định dạng đồng nhất và bổ sung các trường đặc trưng mới.



| Phần | Nội dung |
|---|---|
| 0 | Nạp dữ liệu & chẩn đoán ban đầu |
| 1 | Baseline trên dữ liệu thô (mốc so sánh) |
| 2 | Loại cột leakage & zero-variance |
| 3 | Trích xuất đặc trưng từ text |
| 4 | Chuẩn hoá `loai_hinh` |
| 5 | Khử trùng lặp |
| 6 | Lọc outlier hai tầng |
| 7 | Điền khuyết có kiểm soát |
| 8 | Chặn leakage trong text |
| 9 | Feature phái sinh & xuất file |


**Kết quả:** <br>
Độ sạch và Toàn vẹn:

Loại bỏ hoàn toàn các trường dữ liệu nhiễu, ký tự đặc biệt không hợp lệ và các bản ghi trùng lặp, đảm bảo độ tin cậy tuyệt đối cho các trường định danh và văn bản chính.

Giảm thiểu tối đa tỷ lệ giá trị thiếu (missing values) thông qua các phương pháp điền dữ liệu thông minh hoặc loại bỏ có chọn lọc, bảo toàn quy mô mẫu tối ưu nhất cho tập dữ liệu.

Tính nhất quán (Consistency):

Định dạng dữ liệu giữa các cột đã được đồng bộ hóa hoàn toàn (về kiểu dữ liệu, định dạng chuỗi, khoảng trắng và bảng mã), giải quyết triệt để tình trạng lệch pha dữ liệu.

Tính sẵn sàng cho mô hình (Model Readiness):

Bảng dữ liệu df_clean sau khi bổ sung các đặc trưng mới từ tiêu đề và mô tả đã đạt trạng thái tối ưu, sẵn sàng tích hợp trực tiếp vào các mô hình Học máy (Machine Learning), thuật toán Xử lý ngôn ngữ tự nhiên (NLP) hoặc các hệ thống trực quan hóa dữ liệu (BI Dashboards) mà không cần qua các bước xử lý trung gian nào khác.

## 0. Nạp dữ liệu & chẩn đoán ban đầu

In [2]:
import pandas as pd, numpy as np, re, json, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

SRC = "C:\\Users\\hatro\\Downloads\\du-doan-gia-nha-dat-hcm-HuuDuy-s-Branch\\du-doan-gia-nha-dat-hcm\\data\\test(non_use_file)\\df_raw.csv"   # <-- doi duong dan neu can
raw = pd.read_csv(SRC)
print(raw.shape)
raw.head(3)

(11888, 10)


,tieu_de,gia_ban,dien_tich_m2,tinh_thanh,quan_huyen,mo_ta_dac_diem,loai_hinh,nguon_du_lieu,url,gia_moi_m2_trieu
0,"Bán căn hộ 2PN 2WC 74,8m2 full NT The Sun Aven...",7000.0,74.8,TP. Hồ Chí Minh,Quận 2,"Căn hộ chung cư tại The Sun Avenue, Mai Chí Th...",căn hộ,cafeland,https://nhadat.cafeland.vn/ban-can-ho-2pn-2wc-...,93.582888
1,Bán nhà riêng: Bán nhà 100m2- 6.98 tỷ tại Vo V...,6980.0,100.0,TP. Hồ Chí Minh,Quận 6,"Bán nhà riêng An Lạc, TP. Hồ Chí Minh HẠ GIÁ G...",Nhà rieng,cafeland,https://nhadat.cafeland.vn/ban-nha-rieng-ban-n...,69.800000
2,Bán nhà riêng: Bán nhà mặt tiền đường Nhánh An...,8900.0,90.0,TP. Hồ Chí Minh,Quận 6,"Bán nhà riêng An Lạc, TP. Hồ Chí Minh Bán nhà ...",Nhà rieng,cafeland,https://nhadat.cafeland.vn/ban-nha-rieng-ban-n...,98.888889


In [3]:
print('Thieu du lieu:'); print(raw.isna().sum())
print()
print(raw[['gia_ban','dien_tich_m2','gia_moi_m2_trieu']].describe(percentiles=[.01,.25,.5,.75,.99]).round(2))

Thieu du lieu:
tieu_de                0
gia_ban                0
dien_tich_m2           0
tinh_thanh             0
quan_huyen             0
mo_ta_dac_diem         0
loai_hinh           2809
nguon_du_lieu          0
url                    0
gia_moi_m2_trieu       0
dtype: int64

        gia_ban  dien_tich_m2  gia_moi_m2_trieu
count  11888.00      11888.00          11888.00
mean    9568.22         81.01            126.53
std     7655.06         45.20             88.45
min      109.00         10.00              1.18
1%      1000.00         20.00             11.98
25%     4700.00         51.00             74.14
50%     6990.00         70.00            107.69
75%    11900.00        100.00            153.75
99%    38000.00        240.00            460.66
max    44000.00        270.00           2500.00


### Phát hiện 1 — `gia_moi_m2_trieu` là leakage tuyệt đối

Cột này bằng đúng `gia_ban / dien_tich_m2`. Nếu để lại trong X cùng `dien_tich_m2`,
mô hình chỉ cần nhân hai cột là ra nhãn → R² ≈ 1.0 nhưng vô dụng ngoài thực tế.

In [4]:
kiem_tra = (raw.gia_ban / raw.dien_tich_m2 - raw.gia_moi_m2_trieu).abs()
print('Ty le khop tuyet doi:', (kiem_tra < 0.01).mean())

Ty le khop tuyet doi: 1.0


### Phát hiện 2 — outlier, trùng lặp, truncation, lệch nguồn

In [5]:
ppsm = raw.gia_ban / raw.dien_tich_m2
print('Don gia max         :', round(ppsm.max(),1), 'trieu/m2  <-- phi ly')
print('So dong > 500 tr/m2 :', (ppsm > 500).sum())
print('So dong < 20 tr/m2  :', (ppsm < 20).sum())
print('Trung lap (gia,dt,quan,loai):', raw.duplicated(['gia_ban','dien_tich_m2','quan_huyen','loai_hinh']).sum())
print()
print('Truncation - gia_ban max :', raw.gia_ban.max(), '| so dong > 40000:', (raw.gia_ban>40000).sum())
print('Truncation - dien tich max:', raw.dien_tich_m2.max(), '| so dong > 250:', (raw.dien_tich_m2>250).sum())
print()
print('Lech giua 2 nguon (don gia median):')
print(ppsm.groupby(raw.nguon_du_lieu).median().round(1))

Don gia max         : 2500.0 trieu/m2  <-- phi ly
So dong > 500 tr/m2 : 76
So dong < 20 tr/m2  : 201
Trung lap (gia,dt,quan,loai): 1213

Truncation - gia_ban max : 44000.0 | so dong > 40000: 60
Truncation - dien tich max: 270.0 | so dong > 250: 66

Lech giua 2 nguon (don gia median):
nguon_du_lieu
batdongsan    103.4
cafeland      122.2
dtype: float64


### Phát hiện 3 — `quan_huyen` quá thô, phương sai nội bộ rất lớn

In [6]:
bang = ppsm.groupby(raw.quan_huyen).agg(['count','median','std']).round(1).sort_values('median')
bang

,count,median,std
quan_huyen,,,
Huyện Hóc Môn,307,44.9,142.2
TP. Thủ Đức,741,71.0,58.1
Quận 12,1281,73.4,35.3
Huyện Nhà Bè,130,75.7,30.2
Quận 9,524,80.0,35.3
Quận Bình Tân,877,93.3,36.3
Quận 6,293,103.3,50.1
Quận Tân Phú,859,103.3,42.7
Quận 4,280,111.8,120.9


### Phát hiện 4 — thông tin quyết định giá đang kẹt trong cột text

In [7]:
txt = raw.tieu_de.fillna('') + ' || ' + raw.mo_ta_dac_diem.fillna('')
low = txt.str.lower()
for ten, pat in [('so phong ngu', r'\d+\s*(?:phòng ngủ|pn\b)'), ('so tang', r'\d+\s*(?:tầng|lầu)'),
                 ('so hong/so do', r'sổ hồng|sổ đỏ'), ('trong hem', r'hẻm'),
                 ('ten duong', r'đường\s+\w'), ('CO GIA TRONG TEXT (leak)', r'\d+[\.,]?\d*\s*tỷ')]:
    print(f'{ten:28s}: {low.str.contains(pat, regex=True).mean():.1%}')

so phong ngu                : 78.2%
so tang                     : 73.2%
so hong/so do               : 49.9%
trong hem                   : 44.3%
ten duong                   : 82.6%
CO GIA TRONG TEXT (leak)    : 94.0%


---
# PIPELINE CHUẨN HOÁ

## 2. Loại cột leakage & zero-variance

In [9]:
df = raw.copy()
nhat_ky = {'n_ban_dau': len(df)}

df = df.drop(columns=['gia_moi_m2_trieu',   # leakage: = gia_ban / dien_tich_m2
                      'tinh_thanh'])        # zero variance: chi 1 gia tri
nhat_ky['cot_da_bo'] = ['gia_moi_m2_trieu (leakage)', 'tinh_thanh (zero variance)']
print(df.shape)

(11888, 8)


## 3. Trích xuất đặc trưng từ text

Phần tạo ra phần lớn mức cải thiện. Gộp `tieu_de` + `mo_ta_dac_diem` rồi bóc ra
biến số (phòng ngủ, tầng, mặt tiền, phường, đường) và biến nhị phân (hẻm, sổ hồng, ô tô...).

In [10]:
txt = df.tieu_de.fillna('') + ' || ' + df.mo_ta_dac_diem.fillna('')
low = txt.str.lower()

def so(pattern):
    return pd.to_numeric(low.str.extract(pattern, expand=False), errors='coerce')

# --- bien so ---
df['so_phong_ngu'] = so(r'(\d{1,2})\s*(?:phòng ngủ|pn\b|p\.ngủ|phong ngu)')
df['so_tang']      = so(r'(\d{1,2})\s*(?:tầng|lầu|tang\b|lau\b)')
df['so_wc']        = so(r'(\d{1,2})\s*(?:wc|vệ sinh|toilet)')
df['mat_tien_m']   = so(r'(?:mt|mặt tiền|ngang)[^\d\n]{0,6}(\d{1,2}(?:[.,]\d{1,2})?)\s*m\b')
df['rong_hem_m']   = so(r'hẻm[^\d\n]{0,8}(\d{1,2}(?:[.,]\d{1,2})?)\s*m\b')

# --- bien nhi phan ---
df['la_hem']         = low.str.contains(r'hẻm|hẽm|hem \d').astype(int)
df['la_mat_tien']    = low.str.contains(r'mặt tiền|mặt phố|mt đường').astype(int)
df['co_so_hong']     = low.str.contains(r'sổ hồng|sổ đỏ|shr\b|sở hữu riêng|pháp lý rõ').astype(int)
df['co_noi_that']    = low.str.contains(r'nội thất').astype(int)
df['oto_vao']        = low.str.contains(r'ô tô|oto|xe hơi|ôtô').astype(int)
df['gan_truong_cho'] = low.str.contains(r'trường học|chợ|siêu thị|bệnh viện').astype(int)
df['chinh_chu']      = low.str.contains(r'chính chủ').astype(int)

# --- dia ly chi tiet: ha don vi tu QUAN xuong PHUONG ---
df['phuong'] = (txt.str.extract(
    r'(?:[Pp]hường|\bP\.\s?|\bP\s(?=\d))\s*([A-Za-zÀ-ỹ0-9][A-Za-zÀ-ỹ0-9\s]{0,24}?)'
    r'(?=[,\.\-–\|/]|\s{2}|\s(?:quận|Quận|Q\.|q\.)|$)', expand=False)
    .str.strip().str.title().replace('', np.nan))

df['ten_duong'] = (txt.str.extract(
    r'[ĐđDd]ường\s+([A-ZÀ-Ỹ][A-Za-zÀ-ỹ0-9\s]{1,28}?)(?=[,\.\-–\|]|\s{2}|$)', expand=False)
    .str.strip().str.title().replace('', np.nan))

nhat_ky['ty_le_trich_duoc'] = {c: round(df[c].notna().mean(), 3) for c in
    ['so_phong_ngu','so_tang','so_wc','mat_tien_m','rong_hem_m','phuong','ten_duong']}
pd.Series(nhat_ky['ty_le_trich_duoc']).sort_values(ascending=False)

so_phong_ngu    0.783
so_tang         0.733
ten_duong       0.675
so_wc           0.421
phuong          0.319
mat_tien_m      0.201
rong_hem_m      0.129
dtype: float64

## 4. Chuẩn hoá `loai_hinh`

- `Nhà rieng` → `Nhà riêng` (lỗi chính tả, 2.108 dòng)
- `Nhà hàng - Khách sạn` + `Bán nhà hàng - Khách sạn` + `Kho - Nhà xưởng` → `Khác`
  (mỗi lớp chỉ 2–4 mẫu, không đủ để mô hình học bất cứ điều gì)

In [11]:
anh_xa = {
    'nhà rieng':'Nhà riêng', 'nhà riêng':'Nhà riêng', 'căn hộ':'Căn hộ',
    'nhà phố':'Nhà phố', 'nhà mặt tiền':'Nhà mặt tiền', 'đất':'Đất',
    'biệt thự':'Biệt thự', 'shophouse':'Shophouse',
    'kho - nhà xưởng':'Khác', 'nhà hàng - khách sạn':'Khác', 'bán nhà hàng - khách sạn':'Khác',
}
df['loai_hinh'] = df.loai_hinh.str.lower().str.strip().map(anh_xa).fillna('Khác')
nhat_ky['loai_hinh'] = df.loai_hinh.value_counts().to_dict()
df.loai_hinh.value_counts()

loai_hinh
Nhà riêng       4273
Khác            2822
Căn hộ          1222
Nhà phố         1181
Nhà mặt tiền     989
Đất              859
Biệt thự         494
Shophouse         48
Name: count, dtype: int64

## 5. Khử trùng lặp

URL không trùng, nhưng cùng một căn nhà được đăng trên cả 2 nguồn / bởi nhiều môi giới.
Nếu để lại, bản sao rơi vào cả train và test → điểm test đẹp giả tạo.

In [12]:
truoc = len(df)
df = df.drop_duplicates(subset=['gia_ban','dien_tich_m2','quan_huyen','loai_hinh'], keep='first')
nhat_ky['trung_lap_da_xoa'] = truoc - len(df)
print('Da xoa', nhat_ky['trung_lap_da_xoa'], 'dong trung lap ->', len(df))

Da xoa 1213 dong trung lap -> 10675


## 6. Lọc outlier hai tầng

**Không cắt toàn cục** — Quận 1 và Hóc Môn chênh nhau hàng chục lần, một ngưỡng chung
sẽ xoá nhầm nhà bình thường ở quận đắt.

1. **Chặn cứng theo thị trường:** 15 – 450 triệu/m²
2. **Chặn mềm IQR trong từng quận:** ngoài $[Q_1 - 1.5\,\text{IQR},\ Q_3 + 1.5\,\text{IQR}]$
3. Diện tích ngoài 15 – 500 m²

In [13]:
ppsm = df.gia_ban / df.dien_tich_m2

chan_cung = (ppsm >= 15) & (ppsm <= 450)

nhom = ppsm.groupby(df.quan_huyen)
q1  = nhom.transform(lambda s: s.quantile(.25))
q3  = nhom.transform(lambda s: s.quantile(.75))
iqr = q3 - q1
chan_mem = (ppsm >= q1 - 1.5*iqr) & (ppsm <= q3 + 1.5*iqr)

giu = chan_cung & chan_mem
nhat_ky['outlier_don_gia_da_xoa'] = int((~giu).sum())
df = df[giu].copy()

truoc = len(df)
df = df[(df.dien_tich_m2 >= 15) & (df.dien_tich_m2 <= 500)]
nhat_ky['dien_tich_phi_ly_da_xoa'] = truoc - len(df)

print('Outlier don gia da xoa :', nhat_ky['outlier_don_gia_da_xoa'])
print('Dien tich phi ly da xoa:', nhat_ky['dien_tich_phi_ly_da_xoa'])
print('Con lai:', len(df))

Outlier don gia da xoa : 524
Dien tich phi ly da xoa: 14
Con lai: 10137


## 7. Điền khuyết có kiểm soát

Mỗi kiểu khuyết xử lý theo bản chất của nó, **không điền median bừa**:

| Cột | Cách điền | Lý do |
|---|---|---|
| `so_phong_ngu` | median theo `loai_hinh` | căn hộ và biệt thự phân bố rất khác nhau |
| `so_tang` | `1` | không nhắc tầng ≈ nhà trệt / đất |
| `so_wc` | suy từ số phòng ngủ | tương quan chặt |
| `mat_tien_m`, `rong_hem_m` | **sentinel −1** | "không ghi mặt tiền" tự nó là tín hiệu — điền median sẽ xoá mất |
| `phuong`, `ten_duong` | `Khong_ro` như một hạng mục | |

Thêm 2 cờ **missingness indicator** để mô hình phân biệt giá trị thật vs giá trị điền.

In [14]:
# co MISSINGNESS INDICATOR truoc khi dien
df['co_tt_phong_ngu'] = df.so_phong_ngu.notna().astype(int)
df['co_tt_mat_tien']  = df.mat_tien_m.notna().astype(int)

# clip tran chong loi nhap lieu
for cot, tran in [('so_phong_ngu',10), ('so_tang',8), ('so_wc',8), ('mat_tien_m',30), ('rong_hem_m',15)]:
    df[cot] = df[cot].clip(upper=tran)

df['so_phong_ngu'] = (df.so_phong_ngu
                        .fillna(df.groupby('loai_hinh').so_phong_ngu.transform('median'))
                        .fillna(df.so_phong_ngu.median()))
df['so_tang']    = df.so_tang.fillna(1)
df['so_wc']      = df.so_wc.fillna(df.so_phong_ngu.clip(upper=4))
df['mat_tien_m'] = df.mat_tien_m.fillna(-1)     # sentinel, KHONG dung median
df['rong_hem_m'] = df.rong_hem_m.fillna(-1)     # sentinel
df['phuong']     = df.phuong.fillna('Khong_ro')
df['ten_duong']  = df.ten_duong.fillna('Khong_ro')

print('Con thieu:', df.isna().sum().sum())

Con thieu: 0


## 8. Chặn leakage trong text

**84,5% mô tả gốc chứa giá bán viết thẳng trong nội dung** ("Giá 9.8 tỷ", "3 tỷ 300 triệu").
Nếu TF-IDF / embedding cột gốc, mô hình đọc trộm đáp án.

Thay mọi chuỗi số kèm đơn vị bằng token `<SO>`, bỏ hai cột gốc.

In [15]:
mau_so = re.compile(r'\d+[\.,]?\d*\s*(tỷ|ty|triệu|trieu|tr/m2|tr\b|m²|m2)', re.I)
df['mo_ta_sach']  = df.mo_ta_dac_diem.fillna('').str.replace(mau_so, ' <SO> ', regex=True)
df['tieu_de_sach'] = df.tieu_de.fillna('').str.replace(mau_so, ' <SO> ', regex=True)

print('TRUOC:', df.mo_ta_dac_diem.iloc[0][:180])
print()
print('SAU  :', df.mo_ta_sach.iloc[0][:180])

TRUOC: Căn hộ chung cư tại The Sun Avenue, Mai Chí Thọ, An Phú, Quận 2, Hồ Chí Minh đang chờ chủ mới. Với diện tích 74,8m2 gồm 2PN + 2WC phù hợp cho những ai yêu thích không gian rộng rãi

SAU  : Căn hộ chung cư tại The Sun Avenue, Mai Chí Thọ, An Phú, Quận 2, Hồ Chí Minh đang chờ chủ mới. Với diện tích  <SO>  gồm 2PN + 2WC phù hợp cho những ai yêu thích không gian rộng rãi


## 9. Feature phái sinh & xuất file

- `log_gia_ban` — target lệch phải nặng; train trên thang log rồi `exp()` ngược giảm ~10 điểm MAPE
- `log_dien_tich` — quan hệ giá–diện tích là phi tuyến
- `mat_do_xay` = số tầng × diện tích ≈ diện tích sàn sử dụng

In [ ]:
df['log_gia_ban']   = np.log(df.gia_ban)
df['log_dien_tich']  = np.log(df.dien_tich_m2)
df['mat_do_xay']     = df.so_tang * df.dien_tich_m2

THU_TU_COT = ['gia_ban','log_gia_ban','dien_tich_m2','log_dien_tich',
              'quan_huyen','phuong','ten_duong','loai_hinh',
              'so_phong_ngu','so_tang','so_wc','mat_tien_m','rong_hem_m','mat_do_xay',
              'la_hem','la_mat_tien','co_so_hong','co_noi_that','oto_vao',
              'gan_truong_cho','chinh_chu','co_tt_phong_ngu','co_tt_mat_tien',
              'nguon_du_lieu','tieu_de_sach','mo_ta_sach','url']
df = df[THU_TU_COT]

nhat_ky['n_con_lai']   = len(df)
nhat_ky['ty_le_giu_lai'] = round(len(df) / nhat_ky['n_ban_dau'], 3)
print(json.dumps(nhat_ky, ensure_ascii=False, indent=1))

df.to_csv('df_clean.csv',index=False)  
print('đã lưu df_clean.csv')

{
 "n_ban_dau": 11888,
 "cot_da_bo": [
  "gia_moi_m2_trieu (leakage)",
  "tinh_thanh (zero variance)"
 ],
 "ty_le_trich_duoc": {
  "so_phong_ngu": 0.783,
  "so_tang": 0.733,
  "so_wc": 0.421,
  "mat_tien_m": 0.201,
  "rong_hem_m": 0.129,
  "phuong": 0.319,
  "ten_duong": 0.675
 },
 "loai_hinh": {
  "Nhà riêng": 4273,
  "Khác": 2822,
  "Căn hộ": 1222,
  "Nhà phố": 1181,
  "Nhà mặt tiền": 989,
  "Đất": 859,
  "Biệt thự": 494,
  "Shophouse": 48
 },
 "trung_lap_da_xoa": 1213,
 "outlier_don_gia_da_xoa": 524,
 "dien_tich_phi_ly_da_xoa": 14,
 "n_con_lai": 10137,
 "ty_le_giu_lai": 0.853
}


In [17]:
df.head(5)

,gia_ban,log_gia_ban,dien_tich_m2,log_dien_tich,quan_huyen,phuong,ten_duong,loai_hinh,so_phong_ngu,so_tang,...,co_noi_that,oto_vao,gan_truong_cho,chinh_chu,co_tt_phong_ngu,co_tt_mat_tien,nguon_du_lieu,tieu_de_sach,mo_ta_sach,url
0,7000.0,8.853665,74.8,4.314818,Quận 2,Khong_ro,Khong_ro,Căn hộ,2.0,1.0,...,1,0,0,0,1,0,cafeland,Bán căn hộ 2PN 2WC <SO> full NT The Sun Aven...,"Căn hộ chung cư tại The Sun Avenue, Mai Chí Th...",https://nhadat.cafeland.vn/ban-can-ho-2pn-2wc-...
1,6980.0,8.850804,100.0,4.605170,Quận 6,An Lạc,Khong_ro,Nhà riêng,4.0,3.0,...,1,1,0,1,1,0,cafeland,Bán nhà riêng: Bán nhà <SO> - <SO> tại Vo V...,"Bán nhà riêng An Lạc, TP. Hồ Chí Minh HẠ GIÁ G...",https://nhadat.cafeland.vn/ban-nha-rieng-ban-n...
2,8900.0,9.093807,90.0,4.499810,Quận 6,Khong_ro,Đang Mở Rộng,Nhà riêng,5.0,3.0,...,0,0,0,0,1,1,cafeland,Bán nhà riêng: Bán nhà mặt tiền đường Nhánh An...,"Bán nhà riêng An Lạc, TP. Hồ Chí Minh Bán nhà ...",https://nhadat.cafeland.vn/ban-nha-rieng-ban-n...
3,5200.0,8.556414,43.0,3.761200,Quận Bình Tân,Khong_ro,Khong_ro,Nhà riêng,4.0,2.0,...,0,1,0,0,1,1,cafeland,Bán nhà riêng: KHU VIP TÊN LỬA – NHÀ NGANG 6.5...,"Bán nhà riêng An Lạc, TP. Hồ Chí Minh Hàng hiế...",https://nhadat.cafeland.vn/ban-nha-rieng-khu-v...
4,5800.0,8.665613,60.0,4.094345,Quận Bình Tân,An Lạc Tp,Khong_ro,Nhà riêng,3.0,3.0,...,0,0,0,0,1,1,cafeland,Bán nhà riêng: Bán Nhà Mới <SO> giá <SO> t...,"Bán nhà riêng An Lạc, TP. Hồ Chí Minh Bán Nhà ...",https://nhadat.cafeland.vn/ban-nha-rieng-ban-n...


### Feature importance — kiểm tra xem đặc trưng mới có thực sự đóng góp không

---
## Hạn chế còn tồn tại

Bốn điểm dưới đây **không sửa được bằng tiền xử lý**, cần bổ sung dữ liệu:

1. **`phuong` chỉ trích được ~34%** — 2/3 số tin không ghi phường. Giới hạn của nguồn, không phải của regex.
2. **Truncation vẫn còn:** `gia_ban` max = 43.500 và `dien_tich_m2` max = 270 đúng bằng ngưỡng lọc của bản gốc.
   Mô hình **không hợp lệ** cho BĐS > 43,5 tỷ hoặc > 270 m² — phải nêu rõ phạm vi áp dụng khi báo cáo.
3. **Không có toạ độ địa lý.** Muốn xuống dưới 20% MAPE gần như bắt buộc phải geocode
   `ten_duong` + `phuong` ra lat/lng rồi tính khoảng cách tới trung tâm Quận 1, tới metro, trường, chợ.
4. **Không có mốc thời gian đăng tin** — nếu dữ liệu trải dài nhiều tháng thì có drift giá thị trường
   mà mô hình không nhìn thấy.

### Bước tiếp theo nên làm

- Thay RandomForest bằng **LightGBM / XGBoost** với `objective='mae'` hoặc Huber trên thang log
- **Target-encode `ten_duong`** bằng K-fold (cardinality cao, one-hot sẽ nổ chiều)
- Báo cáo **cả MAPE và MedAPE** — MAPE bị vài outlier thao túng, MedAPE phản ánh chất lượng thực tế hơn
- Cân nhắc mô hình riêng cho `Đất` (464 dòng) — không có tầng/phòng, cơ chế định giá khác hẳn nhà xây
